In [24]:
import numpy as np

In [25]:
tokens = ["I", "love", "AI"]

In [26]:
embeddings = np.array([
    [1.0, 2.0, 3.0, 4.0], # I
    [5.0, 6.0, 7.0, 8.0], # Love
    [9.0, 10.0, 11.0, 12.0] # AI
])

## Sinusoidal Positional Encoding

$$
PE_{pos,2i} =
\sin\left(
\frac{pos}{10000^{\frac{2i}{d_{model}}}}
\right)
$$

$$
PE_{pos,2i+1} =
\cos\left(
\frac{pos}{10000^{\frac{2i}{d_{model}}}}
\right)
$$



In [30]:
def positional_encoding(seq_len, d_model):
    PE = np.zeros_like(embeddings)
    for pos in range(0, seq_len):
        for i in range(0, d_model, 2):
            theta = pos / (10000 ** (i / d_model))
            PE[pos, i] = np.sin(theta)
            PE[pos, i + 1] = np.cos(theta)
    return PE


PE = positional_encoding(seq_len, d_model)
X_pos = embeddings + PE



for i, token in enumerate(tokens):
    print(f"{token:5} Embedding: {embeddings[i]}")
    print(f"      PE:        {PE[i]}")
    print(f"      Input:     {X_pos[i]}")
    print()

I     Embedding: [1. 2. 3. 4.]
      PE:        [0. 1. 0. 1.]
      Input:     [1. 3. 3. 5.]

love  Embedding: [5. 6. 7. 8.]
      PE:        [0.84147098 0.54030231 0.00999983 0.99995   ]
      Input:     [5.84147098 6.54030231 7.00999983 8.99995   ]

AI    Embedding: [ 9. 10. 11. 12.]
      PE:        [ 0.90929743 -0.41614684  0.01999867  0.99980001]
      Input:     [ 9.90929743  9.58385316 11.01999867 12.99980001]



## RoPE

$$
\theta_{pos,i}
=
\frac{pos}{10000^{\frac{2i}{d_{model}}}}
$$



### Matrix Form

$$
\begin{bmatrix}
x'_0\\
x'_1
\end{bmatrix}
=
\begin{bmatrix}
\cos\theta & -\sin\theta\\
\sin\theta & \cos\theta
\end{bmatrix}
\begin{bmatrix}
x_0\\
x_1
\end{bmatrix}
$$

$$
x'_0 = x_0\cos(\theta) - x_1\sin(\theta)
$$

$$
x'_1 = x_0\sin(\theta) + x_1\cos(\theta)
$$

In [31]:
def apply_rope(x):
    seq_len, d_model = x.shape
    output = np.zeros_like(x)
    for pos in range(0, seq_len):
        for i in range(0, d_model, 2):
             theta = pos / (10000 ** (i / d_model))
             cos = np.cos(theta)
             sin = np.sin(theta)

             x0 = x[pos, i]
             x1 = x[pos, i + 1]

             output[pos, i] =  x0 * cos - x1 * sin
             output[pos, i + 1] =  x0 * sin + x1 * cos

    return output
                
X_rope = apply_rope(embeddings)

In [32]:
for i, token in enumerate(tokens):
    print(f"{token:5} Embedding: {embeddings[i]}")
    print(f"      Input:     {X_rope[i]}")
    print()

I     Embedding: [1. 2. 3. 4.]
      Input:     [1. 2. 3. 4.]

love  Embedding: [5. 6. 7. 8.]
      Input:     [-2.34731438  7.44916876  6.91965134  8.06959884]

AI    Embedding: [ 9. 10. 11. 12.]
      Input:     [-12.8382958    4.02220848  10.75781607  12.21758541]

